In [44]:
import pandas as pd
data = pd.read_csv('../Data/preprocessed_data.csv',)

In [45]:
data.isnull().sum()

timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
month                        0
weekday                      0
hour                         0
dtype: int64

### Splitting data Chronologically- Temporal Splitting

In [46]:
data = data.sort_values('timestamp').reset_index(drop = True)

categorical_cols = data.select_dtypes(include = 'object').columns
numerical_cols = data.select_dtypes(include = ['int64', 'float64']).drop(columns = ['is_fraud']).columns

Split_index = int(len(data)*0.8)
train_data  = data.iloc[:Split_index]
test_data = data.iloc[Split_index:]


x_train = train_data.drop('is_fraud', axis = 1)
y_train = train_data['is_fraud']
x_test = test_data.drop('is_fraud',axis = 1)
y_test = test_data['is_fraud']



C:\Users\admin\AppData\Local\Temp\ipykernel_34256\1581499148.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = data.select_dtypes(include = 'object').columns


### Standardization and Encoding

In [47]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers =[
    ('cat', OneHotEncoder(drop = 'first', handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numerical_cols)])

x_train_processed= preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)    

C:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


### Train a Base Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train logistic regression model with class_weight = 'balanced' to handle class imbalance
lr_model = LogisticRegression(max_iter = 1000, random_state = 42, class_weight = 'balanced')
lr_model.fit(x_train_processed, y_train)

# predict
y_pred= lr_model.predict(x_test_processed)

# evaluate model
print(f"Classification report:/n{classification_report(y_test, y_pred)}")
print(f"Confusion matrix:/n{confusion_matrix(y_test, y_pred)}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred)}")
            

Classification report:/n              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1860
           1       0.93      0.92      0.93       308

    accuracy                           0.98      2168
   macro avg       0.96      0.96      0.96      2168
weighted avg       0.98      0.98      0.98      2168

Confusion matrix:/n[[1838   22]
 [  24  284]]
ROC AUC Score: 0.9551249825443374
